# Proyecto Final Big Data — Siniestralidad vial en Bogotá
## Cuaderno 3 · Almacenamiento analítico: CockroachDB (NewSQL) y MongoDB Atlas (NoSQL)

**Entrada:** `interim_parquet.zip` (salida del cuaderno 1). **Salida:** `resultados_cuaderno3.zip`.

**Por qué dos motores (decisión de diseño):**
- **CockroachDB (NewSQL)** guarda el modelo dimensional: tabla de hechos y dimensiones con llaves foráneas, consultas con `JOIN`, funciones de ventana y plan de ejecución (`EXPLAIN`).
- **MongoDB Atlas (NoSQL)** guarda un documento por siniestro con la ubicación como GeoJSON y un índice `2dsphere`, para consultas geoespaciales (cercanía a un punto, zonas calientes por celdas).

Al final se compara el conteo por gravedad en los tres lugares (Parquet, CockroachDB y MongoDB) para evidenciar consistencia.

**Credenciales:** se leen de los *Secretos* de Colab (`ATLAS_URI`, `COCKROACH_URL` y, opcional, `COCKROACH_CERT_URL` con la URL del certificado del clúster; acceso al cuaderno activado). Nunca se imprimen ni se guardan en el cuaderno.

**Requisito de red:** en Atlas, *Network Access* debe permitir la IP de Colab (que cambia en cada sesión); durante la ejecución suele ser necesario `0.0.0.0/0`, y conviene quitarlo después.

## 0. Dependencias y entradas

In [ ]:
!pip -q install "pymongo[srv]" psycopg2-binary

In [ ]:
import zipfile, time
from pathlib import Path

BASE = Path('/content/proyecto_bigdata')
OUT = BASE / 'out'
for d in (BASE, OUT):
    d.mkdir(parents=True, exist_ok=True)

if not (BASE / 'interim' / 'fact_siniestros').exists():
    zips = [p for p in (Path('/content/interim_parquet.zip'), Path('interim_parquet.zip')) if p.exists()]
    if not zips:
        raise FileNotFoundError(
            'Falta interim_parquet.zip (se descarga al final del cuaderno 1). Arrástralo al panel de Archivos '
            '(izquierda, carpeta /content) y vuelve a ejecutar esta celda. Evita files.upload(): es muy lento.')
    zipfile.ZipFile(zips[0]).extractall(BASE)
    print('Parquet extraído desde', zips[0])

METRICAS = []          # (motor, operación, segundos)
def cronometrar(motor, operacion):
    class _C:
        def __enter__(self): self.t0 = time.perf_counter(); return self
        def __exit__(self, *a): METRICAS.append((motor, operacion, round(time.perf_counter() - self.t0, 2)))
    return _C()
print('Listo')

## 1. Credenciales y conexión

In [ ]:
from google.colab import userdata, files
from getpass import getpass

def secreto(nombre):
    try:
        return userdata.get(nombre)
    except Exception:
        return getpass(f'{nombre} (no aparece en pantalla): ')

ATLAS_URI = secreto('ATLAS_URI')
COCKROACH_URL = secreto('COCKROACH_URL')

# Certificado raíz del clúster de CockroachDB (el que indica la consola: .../clusters/<id>/cert).
# Opción A: guardar esa URL en el secreto COCKROACH_CERT_URL y el cuaderno lo descarga solo.
# Opción B: ejecutar antes, en una celda:  !curl --create-dirs -o $HOME/.postgresql/root.crt '<URL del cert>'
import requests
RUTA_CERT = Path.home() / '.postgresql' / 'root.crt'
try:
    CERT_URL = userdata.get('COCKROACH_CERT_URL')
except Exception:
    CERT_URL = None
if CERT_URL and not RUTA_CERT.exists():
    RUTA_CERT.parent.mkdir(parents=True, exist_ok=True)
    resp = requests.get(CERT_URL, timeout=30)
    resp.raise_for_status()
    RUTA_CERT.write_bytes(resp.content)
    print('Certificado del clúster descargado')
print('root.crt presente:', RUTA_CERT.exists())

In [ ]:
import psycopg2
from pymongo import MongoClient

SCHEMA = 'siniestros_bogota'

from urllib.parse import urlparse, parse_qsl, urlencode

def url_sql(modo_ssl=None):
    # Convierte la URL del secreto a formato libpq. Si existe ~/.postgresql/root.crt, libpq lo usa solo.
    p = urlparse(COCKROACH_URL.replace('cockroachdb://', 'postgresql://', 1))
    q = dict(parse_qsl(p.query, keep_blank_values=True))
    if modo_ssl:
        q['sslmode'] = modo_ssl
        q.pop('sslrootcert', None)
    elif q.get('sslmode', '').startswith('verify') and 'sslrootcert' not in q and not RUTA_CERT.exists():
        q['sslrootcert'] = 'system'     # sin root.crt local: intenta con los certificados del sistema
    return p._replace(query=urlencode(q)).geturl()

def conectar_sql():
    try:
        conn = psycopg2.connect(url_sql())
    except psycopg2.OperationalError as e:
        if 'certificate' not in str(e).lower():
            raise
        print('Aviso: no se pudo verificar el certificado del servidor; se conecta cifrado pero SIN verificar (sslmode=require).')
        conn = psycopg2.connect(url_sql('require'))
    conn.autocommit = True
    return conn

conn = conectar_sql()
cur = conn.cursor()
cur.execute('SELECT version()')
print('SQL :', cur.fetchone()[0][:80])



### Conexión a MongoDB Atlas
Si falla con `SSL handshake failed`, casi siempre es la lista de IP de Atlas (*Network Access*) o un clúster pausado.

In [ ]:
import certifi

def conectar_mongo():
    cliente = MongoClient(ATLAS_URI, serverSelectionTimeoutMS=20000, tlsCAFile=certifi.where())
    try:
        cliente.admin.command('ping')
    except Exception:
        try:
            ip = requests.get('https://api.ipify.org', timeout=10).text
        except Exception:
            ip = '(no se pudo obtener)'
        print('No se pudo conectar a Atlas. Causas más comunes:')
        print('  1) Network Access de Atlas no permite esta IP (agrega 0.0.0.0/0 o la IP de abajo y espera ~1 min)')
        print('  2) El clúster gratuito está pausado (reactívalo desde el panel de Atlas)')
        print('  3) Usuario o contraseña incorrectos en ATLAS_URI')
        print('IP pública de esta sesión de Colab:', ip)
        raise
    return cliente

mongo = conectar_mongo()
db = mongo['siniestros_bogota']
try:
    print('Mongo: versión', mongo.server_info()['version'])
except Exception:
    print('Mongo: conectado')

## 2. Carga de la capa limpia (Parquet) a memoria

In [ ]:
import pandas as pd

I = BASE / 'interim'
hechos = pd.read_parquet(I / 'fact_siniestros')
dim_loc = pd.read_parquet(I / 'dim_localidad')
dim_cla = pd.read_parquet(I / 'dim_clase')
dim_gra = pd.read_parquet(I / 'dim_gravedad')
dim_tie = pd.read_parquet(I / 'dim_tiempo')

hechos['anio'] = hechos['anio'].astype(int)
for c in ('mes', 'hora', 'localidad_id', 'clase_id', 'gravedad_id', 'civ', 'pk_calzada'):
    hechos[c] = pd.to_numeric(hechos[c], errors='coerce').astype('Int64')   # enteros con nulos, sin 123.0
hechos['fecha_hora'] = pd.to_datetime(hechos['fecha_hora'])
dim_tie = dim_tie.dropna(subset=['fecha'])

# Los registros sin localidad/clase/gravedad se asignan a una fila 'SIN DATO' de la dimensión
def completar_dim(dim, nombre, id_col, extra=None):
    dim = dim.copy()
    dim[nombre] = dim[nombre].fillna('SIN DATO')
    n_nulos = int(hechos[id_col].isna().sum())
    if n_nulos:
        previo = dim.loc[dim[nombre] == 'SIN DATO', id_col]
        if len(previo):
            nuevo = int(previo.iloc[0])
        else:
            nuevo = int(dim[id_col].max()) + 1
            fila = {id_col: nuevo, nombre: 'SIN DATO', **(extra or {})}
            dim = pd.concat([dim, pd.DataFrame([fila])], ignore_index=True)
        hechos[id_col] = hechos[id_col].fillna(nuevo)
    print(f'{nombre}: {n_nulos} hechos sin {id_col} asignados a SIN DATO')
    return dim

dim_loc = completar_dim(dim_loc, 'localidad', 'localidad_id')
dim_cla = completar_dim(dim_cla, 'clase_acc', 'clase_id')
dim_gra = completar_dim(dim_gra, 'gravedad', 'gravedad_id', extra={'nivel': 0})
print('hechos:', hechos.shape, '| localidades:', len(dim_loc), '| clases:', len(dim_cla),
      '| gravedades:', len(dim_gra), '| fechas:', len(dim_tie))

---
# CockroachDB (NewSQL): modelo dimensional
## 3. Esquema (tabla de hechos + dimensiones con llaves foráneas)
Se recrea el esquema `siniestros_bogota` en cada ejecución (solo borra ese esquema).

In [ ]:
DDL = f'''
DROP SCHEMA IF EXISTS {SCHEMA} CASCADE;
CREATE SCHEMA {SCHEMA};

CREATE TABLE {SCHEMA}.dim_localidad (localidad_id INT PRIMARY KEY, localidad TEXT NOT NULL);
CREATE TABLE {SCHEMA}.dim_clase     (clase_id INT PRIMARY KEY, clase_acc TEXT NOT NULL);
CREATE TABLE {SCHEMA}.dim_gravedad  (gravedad_id INT PRIMARY KEY, gravedad TEXT NOT NULL, nivel INT NOT NULL);
CREATE TABLE {SCHEMA}.dim_tiempo (
    fecha DATE PRIMARY KEY, anio INT NOT NULL, mes INT NOT NULL,
    fin_de_semana BOOL NOT NULL, dia_semana TEXT NOT NULL);

CREATE TABLE {SCHEMA}.fact_siniestros (
    codigo_accidente BIGINT PRIMARY KEY,
    formulario       TEXT,
    fecha_hora       TIMESTAMP,
    fecha            DATE REFERENCES {SCHEMA}.dim_tiempo (fecha),
    anio             INT,
    mes              INT,
    hora             INT,
    franja           TEXT,
    localidad_id     INT REFERENCES {SCHEMA}.dim_localidad (localidad_id),
    clase_id         INT REFERENCES {SCHEMA}.dim_clase (clase_id),
    gravedad_id      INT REFERENCES {SCHEMA}.dim_gravedad (gravedad_id),
    es_grave         BOOL,
    direccion        TEXT,
    latitud          DOUBLE PRECISION,
    longitud         DOUBLE PRECISION,
    coord_valida     BOOL,
    civ              BIGINT,
    pk_calzada       BIGINT
);
'''
for stmt in [s.strip() for s in DDL.split(';') if s.strip()]:
    cur.execute(stmt)
print('Esquema creado')

## 4. Carga masiva (por lotes)

In [ ]:
from psycopg2.extras import execute_values

def filas(df, cols):
    sub = df[cols].astype(object)
    sub = sub.where(df[cols].notna(), None)
    return [tuple(r) for r in sub.itertuples(index=False, name=None)]

def cargar(tabla, df, cols, pagina=5000):
    datos = filas(df, cols)
    sql = f'INSERT INTO {SCHEMA}.{tabla} ({", ".join(cols)}) VALUES %s'
    with cronometrar('CockroachDB', f'carga {tabla} ({len(datos)} filas)'):
        execute_values(cur, sql, datos, page_size=pagina)
    return len(datos)

cargar('dim_localidad', dim_loc, ['localidad_id', 'localidad'])
cargar('dim_clase', dim_cla, ['clase_id', 'clase_acc'])
cargar('dim_gravedad', dim_gra, ['gravedad_id', 'gravedad', 'nivel'])
cargar('dim_tiempo', dim_tie, ['fecha', 'anio', 'mes', 'fin_de_semana', 'dia_semana'])

h = hechos.copy()
h['fecha_hora'] = h['fecha_hora'].dt.to_pydatetime()
cols_fact = ['codigo_accidente', 'formulario', 'fecha_hora', 'fecha', 'anio', 'mes', 'hora', 'franja',
             'localidad_id', 'clase_id', 'gravedad_id', 'es_grave', 'direccion', 'latitud', 'longitud',
             'coord_valida', 'civ', 'pk_calzada']
n_fact = cargar('fact_siniestros', h, cols_fact)

with cronometrar('CockroachDB', 'índice secundario (localidad, año)'):
    cur.execute(f'CREATE INDEX idx_fact_loc_anio ON {SCHEMA}.fact_siniestros (localidad_id, anio)')

cur.execute(f'SELECT count(*) FROM {SCHEMA}.fact_siniestros')
print('filas en fact_siniestros:', cur.fetchone()[0], '| esperadas:', len(hechos))

## 5. Consultas analíticas en SQL

In [ ]:
def q(sql, motor='CockroachDB', nombre=None):
    with cronometrar(motor, nombre or sql.strip().split('\n')[0][:60]):
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        filas_ = cur.fetchall()
    return pd.DataFrame(filas_, columns=cols)

S = SCHEMA
total = len(hechos)
min_n = 500 if total > 5000 else 1

sql_resultados = {}

sql_resultados['sql_anio_gravedad'] = q(f'''
    SELECT f.anio, g.gravedad, count(*) AS siniestros
    FROM {S}.fact_siniestros f JOIN {S}.dim_gravedad g USING (gravedad_id)
    GROUP BY f.anio, g.gravedad ORDER BY f.anio, g.gravedad''', nombre='año x gravedad (JOIN)')

sql_resultados['sql_variacion_mensual'] = q(f'''
    WITH m AS (SELECT anio, mes, count(*) AS n FROM {S}.fact_siniestros GROUP BY anio, mes)
    SELECT anio, mes, n, n - LAG(n) OVER (ORDER BY anio, mes) AS var_vs_mes_anterior
    FROM m ORDER BY anio, mes''', nombre='variación mensual (LAG)')

sql_resultados['sql_localidades_grave'] = q(f'''
    SELECT l.localidad, count(*) AS siniestros,
           round(100.0 * avg(f.es_grave::int), 1) AS pct_graves,
           rank() OVER (ORDER BY avg(f.es_grave::int) DESC) AS ranking
    FROM {S}.fact_siniestros f JOIN {S}.dim_localidad l USING (localidad_id)
    GROUP BY l.localidad HAVING count(*) >= {min_n}
    ORDER BY ranking LIMIT 10''', nombre='ranking localidades % graves (rank)')

sql_resultados['sql_franja_muertos'] = q(f'''
    SELECT f.franja, count(*) AS siniestros,
           round(100.0 * sum((g.nivel = 2)::int) / count(*), 2) AS pct_con_muertos
    FROM {S}.fact_siniestros f JOIN {S}.dim_gravedad g USING (gravedad_id)
    GROUP BY f.franja ORDER BY pct_con_muertos DESC''', nombre='% con muertos por franja')

for nombre, df_ in sql_resultados.items():
    print('\n###', nombre)
    display(df_.head(15))

### Plan de ejecución (`EXPLAIN`) de una consulta filtrada por localidad y año

In [ ]:
cur.execute(f'EXPLAIN SELECT count(*) FROM {S}.fact_siniestros WHERE localidad_id = 1 AND anio = 2019')
plan = '\n'.join(r[0] for r in cur.fetchall())
print(plan)
(OUT / 'db_explain_cockroach.txt').write_text(plan)

---
# MongoDB Atlas (NoSQL): documentos con GeoJSON
## 6. Documentos (un siniestro = un documento, con la ubicación como `Point`)

In [ ]:
import math

con_texto = (hechos
             .merge(dim_loc, on='localidad_id', how='left')
             .merge(dim_cla, on='clase_id', how='left')
             .merge(dim_gra[['gravedad_id', 'gravedad']], on='gravedad_id', how='left'))

def num(x, tipo):
    return None if pd.isna(x) else tipo(x)

def construir_doc(r):
    d = {
        '_id': int(r.codigo_accidente),
        'formulario': num(r.formulario, str),
        'fecha_hora': None if pd.isna(r.fecha_hora) else r.fecha_hora.to_pydatetime(),
        'anio': num(r.anio, int), 'mes': num(r.mes, int), 'hora': num(r.hora, int),
        'franja': num(r.franja, str), 'localidad': num(r.localidad, str),
        'clase': num(r.clase_acc, str), 'gravedad': num(r.gravedad, str),
        'es_grave': bool(r.es_grave), 'direccion': num(r.direccion, str),
        'civ': num(r.civ, int), 'pk_calzada': num(r.pk_calzada, int),
    }
    if bool(r.coord_valida) and not (pd.isna(r.latitud) or pd.isna(r.longitud)):
        d['ubicacion'] = {'type': 'Point', 'coordinates': [float(r.longitud), float(r.latitud)]}
    return d

docs = [construir_doc(r) for r in con_texto.itertuples(index=False)]
print('documentos:', len(docs), '| con ubicación:', sum('ubicacion' in d for d in docs))
docs[0]

## 7. Carga a Atlas e índices

In [ ]:
col = db['siniestros']
db.drop_collection('siniestros')
col = db['siniestros']

with cronometrar('MongoDB', f'carga siniestros ({len(docs)} docs)'):
    for i in range(0, len(docs), 5000):
        col.insert_many(docs[i:i + 5000], ordered=False)

with cronometrar('MongoDB', 'índices (2dsphere + compuestos)'):
    col.create_index([('ubicacion', '2dsphere')])
    col.create_index([('localidad', 1), ('anio', 1)])
    col.create_index([('anio', 1), ('franja', 1)])

print('documentos en la colección:', col.count_documents({}), '| esperados:', len(docs))
print('índices:', list(col.index_information().keys()))

## 8. Consultas en MongoDB: geoespaciales y de agregación

In [ ]:
PUNTO = [-74.0760, 4.5981]     # [lon, lat]: Plaza de Bolívar (centro de Bogotá)
RADIO_M = 500

def agg(nombre, pipeline):
    with cronometrar('MongoDB', nombre):
        res = list(col.aggregate(pipeline))
    return pd.DataFrame(res)

mongo_resultados = {}

mongo_resultados['mongo_cercanos_gravedad'] = agg(f'siniestros a {RADIO_M} m por gravedad ($geoNear)', [
    {'$geoNear': {'near': {'type': 'Point', 'coordinates': PUNTO}, 'distanceField': 'dist_m',
                  'maxDistance': RADIO_M, 'spherical': True}},
    {'$group': {'_id': '$gravedad', 'siniestros': {'$sum': 1}}},
    {'$sort': {'siniestros': -1}}])

mongo_resultados['mongo_graves_mas_cercanos'] = agg('5 graves más cercanos ($geoNear + filtro)', [
    {'$geoNear': {'near': {'type': 'Point', 'coordinates': PUNTO}, 'distanceField': 'dist_m',
                  'spherical': True, 'query': {'es_grave': True}}},
    {'$limit': 5},
    {'$project': {'_id': 1, 'direccion': 1, 'localidad': 1, 'gravedad': 1, 'anio': 1,
                  'dist_m': {'$round': ['$dist_m', 0]}}}])

mongo_resultados['mongo_muertos_localidad'] = agg('localidades con más siniestros con muertos', [
    {'$match': {'gravedad': 'CON MUERTOS'}},
    {'$group': {'_id': '$localidad', 'con_muertos': {'$sum': 1}}},
    {'$sort': {'con_muertos': -1}}, {'$limit': 10}])

mongo_resultados['mongo_celdas_calientes'] = agg('celdas calientes (~1 km) por conteo', [
    {'$match': {'ubicacion': {'$exists': True}}},
    {'$group': {'_id': {'lat': {'$round': [{'$arrayElemAt': ['$ubicacion.coordinates', 1]}, 2]},
                        'lon': {'$round': [{'$arrayElemAt': ['$ubicacion.coordinates', 0]}, 2]}},
                'siniestros': {'$sum': 1},
                'graves': {'$sum': {'$cond': ['$es_grave', 1, 0]}}}},
    {'$addFields': {'pct_graves': {'$round': [{'$multiply': [{'$divide': ['$graves', '$siniestros']}, 100]}, 1]}}},
    {'$sort': {'siniestros': -1}}, {'$limit': 10}])

for nombre, df_ in mongo_resultados.items():
    print('\n###', nombre)
    display(df_)

## 9. Consistencia entre los tres almacenes (Parquet, CockroachDB, MongoDB)

In [ ]:
p = (hechos.merge(dim_gra[['gravedad_id', 'gravedad']], on='gravedad_id')
     .groupby('gravedad').size().rename('parquet'))
s = q(f'''SELECT g.gravedad, count(*) AS cockroachdb
          FROM {S}.fact_siniestros f JOIN {S}.dim_gravedad g USING (gravedad_id)
          GROUP BY g.gravedad''', nombre='conteo por gravedad (verificación)').set_index('gravedad')['cockroachdb']
m = agg('conteo por gravedad (verificación)', [{'$group': {'_id': '$gravedad', 'mongodb': {'$sum': 1}}}])
m = m.set_index('_id')['mongodb'].rename_axis('gravedad')

consistencia = pd.concat([p, s, m], axis=1)
consistencia.loc['TOTAL'] = consistencia.sum()
consistencia['coinciden'] = consistencia.nunique(axis=1) == 1
consistencia

## 10. Métricas y empaquetado de resultados

In [ ]:
metricas = pd.DataFrame(METRICAS, columns=['motor', 'operacion', 'segundos'])
display(metricas)

metricas.to_csv(OUT / 'db_metricas.csv', index=False)
consistencia.to_csv(OUT / 'db_consistencia.csv')
for nombre, df_ in {**sql_resultados, **mongo_resultados}.items():
    df_.to_csv(OUT / f'db_{nombre}.csv', index=False)

zpath = OUT / 'resultados_cuaderno3.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.glob('db_*')):
        z.write(f, f.name)
print(zpath, f'{zpath.stat().st_size/1e6:.2f} MB')
files.download(str(zpath))

cur.close(); conn.close(); mongo.close()